# Train YOLOv8 on Indian Traffic Data (Colab free T4)

Purpose: train a YOLOv8 detector on Indian CCTV frames, evaluate per-class
mAP50, export ONNX + int8 artifacts, and package them into a model registry
zip ready to drop into the repo's `models/registry/`.

## Data flow (nothing stored locally except the final zip)
1. This notebook pulls the raw subset **directly from Hugging Face**
   (`kalyan1729/trafficmanagementdataset`, IITM-HeTra_v2 slice) into
   Colab's ephemeral `/tmp`.
2. Conversion to the repo contract (`docs/DATASET_SPEC.md` v1) happens
   **in Colab** — same logic as `scripts/prepare_dataset.py --source voc`.
3. Train -> val -> ONNX -> int8 -> registry zip. **Only the zip leaves
   Colab** (browser download, ~50-100 MB). No Drive quota, no local disk.

## Option A pilot (conservative 6-class mapping)
- Contract classes: `car, motorcycle, bus, truck, bicycle, auto`.
- Only exact matches (+ `auto_rickshaw`/`three_wheeler` -> `auto`) are kept;
  `sedan/suv/muv/van/lcv/minibus/tempo_traveller` objects are **dropped**.
- If the `auto` mAP50 gate fails, the follow-up is Option B (aggressive
  merge + BMD-45-Train) — not scope creep inside this run.

## Free T4 strategy
- **Session 1**: `yolov8n` @ 70 epochs, imgsz 640, batch 16. Fits comfortably
  in one free session (~60-80 epochs).
- **Session 2 (optional)**: `yolov8s` @ 50 epochs, batch 12. Run only if the
  n-model mAP is insufficient; it may not fit in one session.

## Runtime instructions
1. `Runtime` -> `Change runtime type` -> **T4 GPU**.
2. `Run all`. No paths to edit, no Drive mount needed for data.

> NOTE: the class schema is PROVISIONAL. Adding classes later only requires
> appending ids at the END of `data.yaml` and retraining — no code changes here.


## Setup

Installs deps, mounts Drive (with a local fallback for structure testing),
and defines `DRIVE_DATASET_ROOT`. Edit the placeholder path before running.

In [1]:
!pip install -q ultralytics onnx onnxruntime huggingface_hub

from huggingface_hub import snapshot_download

# ---- data contract: Option A pilot snapshot (mle-workflow lane: lock it) ----
HF_REPO = "kalyan1729/trafficmanagementdataset"
HF_SUBSET = "IITM-HeTra_v2"  # ~2.8k Pascal-VOC images, smallest Bengaluru slice
ALLOW_PATTERNS = ["IITM-HeTra_v2/**"]

# NOTE: anonymous HF downloads are rate-limited (HTTP 429). If retries stall,
# run `huggingface-cli login` with a free token first, or re-run this cell:
# snapshot_download resumes cached files instead of re-downloading.
RAW_ROOT = snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                             allow_patterns=ALLOW_PATTERNS, max_workers=2)
DRIVE_DATASET_ROOT = "/tmp/india_yolo_dataset"  # contract dataset (ephemeral)

print('raw HF snapshot:', RAW_ROOT)
print('contract root  :', DRIVE_DATASET_ROOT)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5678 files:   0%|          | 0/5678 [00:00<?, ?it/s]

raw HF snapshot: /root/.cache/huggingface/hub/datasets--kalyan1729--trafficmanagementdataset/snapshots/040637b2ba82b364a9f67d22c996102ae47693dd
contract root  : /tmp/india_yolo_dataset


## Convert HF snapshot to contract (in Colab)

Runs the repo's `convert_voc` logic inline (Option A mapping) and samples the int8 calibration set. IITM-HeTra's `trainval.txt`/`test.txt` are honored, so splits stay junction-disjoint per the anti-leakage rule.

In [3]:
import os, glob, shutil, xml.etree.ElementTree as ET
from pathlib import Path

# Mirrors scripts/prepare_dataset.py::convert_voc (Option A). Self-contained:
# Colab has no repo checkout.
CLASSES = ["car", "motorcycle", "bus", "truck", "bicycle", "auto"]

ALIASES = {
    "motorbike": "motorcycle", "moto": "motorcycle",
    "bicycle": "bicycle", "bike": "bicycle",
    "autorickshaw": "auto", "rickshaw": "auto",
    "three_wheeler": "auto", "auto_rickshaw": "auto",
}

def class_index(name):
    n = name.lower().strip().replace(" ", "_").replace("-", "_")
    m = ALIASES.get(n, n)
    return CLASSES.index(m) if m in CLASSES else None

def load_splits(raw_root):
    membership = {}
    for split, files in (("train", ("trainval.txt", "train.txt")),
                          ("val", ("val.txt",)), ("test", ("test.txt",))):
        for lf in files:
            for lp in sorted(Path(raw_root).rglob(lf)):
                for raw in lp.read_text(encoding="utf-8").splitlines():
                    stem = raw.strip().lstrip("\ufeff")
                    if stem:
                        membership[stem] = split
    return membership

def convert_voc(raw_root, out_root):
    xmls = sorted(Path(raw_root).rglob("*.xml"))
    assert xmls, 'no VOC XML found in HF snapshot'
    # HeTra keeps xmls/ and images/ in SEPARATE dirs: index all images by stem.
    img_index = {}
    for pat in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        for q in Path(raw_root).rglob(pat):
            img_index.setdefault(q.stem, q)
    membership = load_splits(raw_root)
    kept = dropped = n_noimg = 0
    for xp in xmls:
        try:
            root = ET.parse(xp).getroot()
        except ET.ParseError:
            continue
        size = root.find('size')
        w, h = int(size.findtext('width')), int(size.findtext('height'))
        if w <= 0 or h <= 0:
            continue
        ip = None
        pt = root.findtext('path')
        if pt and Path(pt).exists():
            ip = Path(pt)
        if ip is None:
            fn = root.findtext('filename') or xp.stem
            for cand in (xp.parent / fn, xp.with_suffix('.jpg'),
                         img_index.get(Path(fn).stem), img_index.get(xp.stem)):
                if cand is not None and cand.exists():
                    ip = Path(cand)
                    break
        if ip is None:
            n_noimg += 1
            continue
        lines = []
        for obj in root.findall('object'):
            idx = class_index(obj.findtext('name') or '')
            if idx is None:
                dropped += 1
                continue
            bb = obj.find('bndbox')
            xmin, ymin = float(bb.findtext('xmin')), float(bb.findtext('ymin'))
            xmax, ymax = float(bb.findtext('xmax')), float(bb.findtext('ymax'))
            bw, bh = xmax - xmin, ymax - ymin
            if bw <= 0 or bh <= 0:
                continue
            lines.append(f"{idx} {(xmin + bw/2)/w:.6f} {(ymin + bh/2)/h:.6f} {bw/w:.6f} {bh/h:.6f}")
        if not lines:
            continue
        split = membership.get(ip.stem, 'train')
        for sub in ('images', 'labels'):
            os.makedirs(os.path.join(out_root, sub, split), exist_ok=True)
        shutil.copy2(ip, os.path.join(out_root, 'images', split, ip.name))
        open(os.path.join(out_root, 'labels', split, ip.stem + '.txt'), 'w').write('\n'.join(lines) + '\n')
        kept += 1
    print(f'voc: wrote {kept} images; dropped {dropped} objects (Option A); {n_noimg} xmls w/o image')
    names = ['path: .', 'train: images/train', 'val: images/val', 'test: images/test', 'names:']
    assert kept > 0, ('voc wrote 0 images; snapshot layout: '
                       + str(sorted(os.listdir(raw_root))[:15]))
    names += [f'  {i}: {n}' for i, n in enumerate(CLASSES)]
    os.makedirs(out_root, exist_ok=True)
    open(os.path.join(out_root, 'data.yaml'), 'w').write('\n'.join(names) + '\n')

# Guard against a stale kernel: if you edited cells without re-running the
# convert_voc definition cell above, fail here instead of running old code.
import inspect
assert 'kept > 0' in inspect.getsource(convert_voc), 'stale kernel: re-run the definition cell above (or Restart and run all)'
# Resolve the subset dir robustly: HF snapshots may differ in case/nesting.
cands = [d for d in Path(RAW_ROOT).rglob('*')
         if d.is_dir() and d.name.lower() == HF_SUBSET.lower()]
print('subset candidates:', [str(d) for d in cands])
assert cands, ('subset ' + HF_SUBSET + ' not found under snapshot; '
               'top level: ' + str([p.name for p in sorted(Path(RAW_ROOT).iterdir())][:15]))
convert_voc(str(cands[0]), DRIVE_DATASET_ROOT)

# calibration frames: 200 random (no captures.json on HF; weather-stratified
# sampling is a documented best-effort gap for the pilot)
import random
rng = random.Random(42)
all_imgs = glob.glob(os.path.join(DRIVE_DATASET_ROOT, 'images', '*', '*.jpg'))
calib_dir = os.path.join(DRIVE_DATASET_ROOT, 'calibration', 'frames')
os.makedirs(calib_dir, exist_ok=True)
for p in rng.sample(all_imgs, min(200, len(all_imgs))):
    shutil.copy2(p, os.path.join(calib_dir, os.path.basename(p)))
print(f'calibration: {len(os.listdir(calib_dir))} frames from {len(all_imgs)} images')

subset candidates: ['/root/.cache/huggingface/hub/datasets--kalyan1729--trafficmanagementdataset/snapshots/040637b2ba82b364a9f67d22c996102ae47693dd/IITM-HeTra_v2']
voc: wrote 2356 images; dropped 6670 objects (Option A); 0 xmls w/o image
calibration: 200 frames from 1178 images


## Validate dataset

Checks images-vs-labels counts per split, asserts exactly 6 classes from
`data.yaml`, warns on empty label files and out-of-range boxes. Self-contained;
no repo imports (Colab has no checkout).

In [4]:
import os, glob

CLASSES = ["car", "motorcycle", "bus", "truck", "bicycle", "auto"]

def validate(root):
    yaml_path = os.path.join(root, 'data.yaml')
    assert os.path.isfile(yaml_path), f'missing {yaml_path}'
    names = {}
    with open(yaml_path) as f:
        in_names = False
        for line in f:
            line = line.strip()
            if line.startswith('names:'):
                in_names = True; continue
            if in_names:
                if ':' in line:
                    k, v = line.split(':', 1)
                    names[int(k.strip())] = v.strip()
                elif not line:
                    break
    assert len(names) == 6, f'expected 6 classes, got {len(names)}'
    assert [names[i] for i in range(6)] == CLASSES,         f'class order must be {CLASSES}, got {[names.get(i) for i in range(6)]}'
    print('data.yaml OK: 6 classes in required order.')

    problems = 0
    for split in ('train', 'val', 'test'):
        imgs = sorted(glob.glob(os.path.join(root, 'images', split, '*.jpg')))
        lbls_dir = os.path.join(root, 'labels', split)
        lbls = {os.path.splitext(os.path.basename(p))[0]: p
                for p in glob.glob(os.path.join(lbls_dir, '*.txt'))}
        empty = 0
        for ip in imgs:
            stem = os.path.splitext(os.path.basename(ip))[0]
            lp = lbls.get(stem)
            if lp is None:
                print(f'WARN [{split}] no label file for {stem}'); problems += 1
                continue
            content = open(lp).read().strip()
            if not content:
                empty += 1
                continue
            for lineno, row in enumerate(content.splitlines(), 1):
                parts = row.split()
                cid = int(parts[0])
                if not (0 <= cid <= 5):
                    print(f'WARN [{split}] {stem}:{lineno} bad class id {cid}'); problems += 1
                vals = [float(x) for x in parts[1:]]
                if any(v < 0 or v > 1 for v in vals):
                    print(f'WARN [{split}] {stem}:{lineno} box outside [0,1]'); problems += 1
        if empty:
            print(f'WARN [{split}] {empty} empty label files')
        print(f'{split}: {len(imgs)} images, {len(lbls)} label files')
    calib = glob.glob(os.path.join(root, 'calibration', 'frames', '*.jpg'))
    print(f'calibration frames: {len(calib)} (want ~200)')
    if len(calib) == 0:
        print('WARN: calibration set missing; int8 quantization will fail later.')
        problems += 1
    print('validation done,', problems, 'problems')

validate(DRIVE_DATASET_ROOT)

data.yaml OK: 6 classes in required order.
train: 987 images, 987 label files
val: 0 images, 0 label files
test: 191 images, 191 label files
calibration frames: 200 (want ~200)
validation done, 0 problems


## Train tier-low model: yolov8n @ 70 epochs

Fits one free T4 session (~60-80 epochs). Prints progress each epoch and a
`results.csv` snippet when done.

In [6]:
import os
root = DRIVE_DATASET_ROOT
names = ["car", "motorcycle", "bus", "truck", "bicycle", "auto"]
def _has(s):
    d = os.path.join(root, 'images', s)
    return os.path.isdir(d) and any(f.endswith('.jpg') for f in os.listdir(d))
val = 'images/val' if _has('val') else ('images/test' if _has('test') else 'images/train')
lines = [f'path: {os.path.abspath(root)}', 'train: images/train',
         f'val: {val}', 'test: images/test', 'names:']
lines += [f'  {i}: {n}' for i, n in enumerate(names)]
open(os.path.join(root, 'data.yaml'), 'w').write('\n'.join(lines) + '\n')
print(open(os.path.join(root, 'data.yaml')).read())

path: /tmp/india_yolo_dataset
train: images/train
val: images/test
test: images/test
names:
  0: car
  1: motorcycle
  2: bus
  3: truck
  4: bicycle
  5: auto



In [7]:
from ultralytics import YOLO

model_n = YOLO('yolov8n.pt')
model_n.train(
    data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'),
    epochs=70, imgsz=640, batch=16, patience=15,
    name='india_yolov8n',
)

run_dir_n = model_n.trainer.save_dir
results_csv = os.path.join(run_dir_n, 'results.csv')
print('\n--- results.csv (head/tail snippet) ---')
with open(results_csv) as f:
    lines = f.readlines()
for ln in lines[:2] + lines[-5:]:
    print(ln.rstrip())
BEST_N = os.path.join(run_dir_n, 'weights', 'best.pt')
print('best.pt:', BEST_N)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/tmp/india_yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=india_yolov8n-2, nbs=64, nms=

## Per-class mAP50 table

Runs validation and prints a formatted table. The `auto` row is highlighted
(`>>>`) because it is the class COCO lacks — watch it closely.

In [8]:
metrics_n = YOLO(BEST_N).val(data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'), verbose=False)

def print_map_table(metrics):
    # ultralytics Metric: .ap_class_index, .p, .r are arrays over present classes;
    # .ap50 is per-class AP@0.5; .map50 is overall scalar
    names = metrics.names  # {id: name}
    idx = metrics.box.ap_class_index
    header = f"{'class':<12} {'precision':>9} {'recall':>9} {'mAP50':>9}"
    print(header)
    print('-' * len(header))
    rows = {}
    for i, ci in enumerate(idx):
        name = names[int(ci)]
        p = float(metrics.box.p[i]); r = float(metrics.box.r[i])
        m = float(metrics.box.ap50[i])
        mark = '>>>' if name == 'auto' else '   '
        print(f"{mark} {name:<9} {p:>9.3f} {r:>9.3f} {m:>9.3f}")
        rows[name] = round(m, 4)
    print(f"\noverall mAP50: {float(metrics.box.map50):.3f}")
    return rows

per_class_n = print_map_table(metrics_n)
mAP50_n = round(float(metrics_n.box.map50), 4)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2465.2±643.0 MB/s, size: 110.1 KB)
val: Scanning /tmp/india_yolo_dataset/labels/test.cache... 191 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 191/191 44.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 3.3it/s 3.6s
                   all        191        518      0.925      0.971      0.982      0.596
Speed: 4.4ms preprocess, 4.1ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detect/val
class        precision    recall     mAP50
------------------------------------------
    car           0.975     0.953     0.990
    bus           0.921     0.960     0.981
>>> auto          0.878     1.000     0.974

overall mAP50: 0.982


## OPTIONAL second session: yolov8s @ 50 epochs

Commented out by default so free-tier users don't blow the session budget.
Uncomment ONLY if yolov8n mAP is insufficient and you have a fresh session.

In [9]:
# model_s = YOLO('yolov8s.pt')
# model_s.train(
#     data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'),
#     epochs=50, imgsz=640, batch=12, patience=15,
#     name='india_yolov8s',
# )
# run_dir_s = model_s.trainer.save_dir
# BEST_S = os.path.join(run_dir_s, 'weights', 'best.pt')
# metrics_s = YOLO(BEST_S).val(data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'), verbose=False)
# per_class_s = print_map_table(metrics_s)
# mAP50_s = round(float(metrics_s.box.map50), 4)

BEST_S = None  # set to best.pt path if you ran the s-model above

## Export ONNX (fp32)

Exports `best.pt` to ONNX (imgsz 640, opset 17, simplified) for every trained
size.

In [10]:
sizes = {'n': BEST_N}
if BEST_S:
    sizes['s'] = BEST_S

onnx_paths = {}
for tag, pt in sizes.items():
    print(f'exporting yolov8{tag} ...')
    onnx_paths[tag] = YOLO(pt).export(format='onnx', imgsz=640, opset=17, simplify=True)
print(onnx_paths)

exporting yolov8n ...
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/detect/india_yolov8n-2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (6.0 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 10 packages in 244ms
Prepared 2 packages in 42ms
Installed 2 packages in 10ms
 + colorama==0.4.6
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 17...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 2.0s,

## int8 static quantization

Calibrates over `calibration/frames/*.jpg` (preprocess: resize 640x640,
BGR->RGB, /255, NCHW float32). Falls back cleanly to dynamic quantization
(static is preferred) if `quantize_static` fails in Colab.

In [11]:
import cv2, numpy as np, traceback
from onnxruntime.quantization import CalibrationDataReader, quantize_static, quantize_dynamic, QuantType

class FrameReader(CalibrationDataReader):
    def __init__(self, frames, onnx_path):
        self.input_name = ort_session_input_name(onnx_path)
        self.reiter = iter([self._pre(p) for p in frames])
    def _pre(self, path):
        img = cv2.imread(path)
        assert img is not None, f'cannot read {path}'
        img = cv2.resize(img, (640, 640))  # simple resize is fine for calibration
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return {self.input_name: np.transpose(rgb, (2, 0, 1))[np.newaxis]}
    def get_next(self):
        return next(self.reiter, None)

def ort_session_input_name(onnx_path):
    import onnxruntime as ort
    return ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider']).get_inputs()[0].name

frames = sorted(glob.glob(os.path.join(DRIVE_DATASET_ROOT, 'calibration', 'frames', '*.jpg')))[:200]
assert frames, 'no calibration frames found'

int8_paths = {}
for tag, onx in onnx_paths.items():
    int8_out = onx.replace('.onnx', '-int8.onnx')
    try:
        reader = FrameReader(frames, onx)
        quantize_static(onx, int8_out, reader, weight_type=QuantType.QInt8)
        print(f'yolov8{tag}: STATIC int8 -> {int8_out}')
    except Exception:
        traceback.print_exc()
        print(f'yolov8{tag}: quantize_static failed, falling back to DYNAMIC '
              '(static is preferred; consider rerunning this cell)')
        quantize_dynamic(onx, int8_out, weight_type=QuantType.QInt8)
    int8_paths[tag] = int8_out
print(int8_paths)

Traceback (most recent call last):
  File "/tmp/ipykernel_14903/2306789833.py", line 29, in <cell line: 0>
    reader = FrameReader(frames, onx)
  File "/tmp/ipykernel_14903/2306789833.py", line 7, in __init__
    self.reiter = iter([self._pre(p) for p in frames])
                        ~~~~~~~~~^^^
  File "/tmp/ipykernel_14903/2306789833.py", line 14, in _pre
    return {self.input_name: np.transpose(rgb, (2, 0, 1))[np.newaxis]}
            ^^^^^^^^^^^^^^^
AttributeError: 'FrameReader' object has no attribute 'input_name'


yolov8n: quantize_static failed, falling back to DYNAMIC (static is preferred; consider rerunning this cell)
{'n': '/content/runs/detect/india_yolov8n-2/weights/best-int8.onnx'}


## Promotion gate (declare before packaging)

Pilot bars for Option A (edit if the team recalibrates): the run only
packages when overall mAP50 clears 0.50 **and** the `auto` class — the
one COCO lacks — clears 0.35. A gate failure means Option B
(aggressive merge + BMD-45), not silent packaging of a weak model.

In [12]:
GATE_OVERALL = 0.50
GATE_AUTO = 0.35

auto_map = per_class_n.get('auto')
print(f'gate: overall mAP50={mAP50_n:.3f} (bar {GATE_OVERALL}), '
      f'auto mAP50={auto_map} (bar {GATE_AUTO})')
assert mAP50_n >= GATE_OVERALL, 'overall mAP50 below pilot bar -> Option B'
assert auto_map is not None and auto_map >= GATE_AUTO, \
    'auto mAP50 below pilot bar -> Option B'
print('gate PASSED: packaging registry artifacts')

gate: overall mAP50=0.982 (bar 0.5), auto mAP50=0.9744 (bar 0.35)
gate PASSED: packaging registry artifacts


## Package model registry

Creates `models/registry/india-yolov8{n,s}/` containing the ONNX models and
`metadata.json` matching the repo contract exactly.

In [13]:
import datetime, json

registry_root = 'models/registry'
os.makedirs(registry_root, exist_ok=True)

def build_meta(name, quant, source_run, map50, per_class):
    return {
        'name': name,
        'classes': CLASSES,
        'imgsz': 640,
        'normalization': {'mean': [0, 0, 0], 'std': [255, 255, 255],
                          'layout': 'NCHW', 'color': 'RGB'},
        'quantization': quant,
        'source_run': source_run,
        'metrics': {'mAP50_overall': map50, 'per_class_mAP50': per_class},
        'provenance': {'hf_repo': HF_REPO, 'hf_subset': HF_SUBSET,
                       'mapping': 'optionA-conservative-6class',
                       'contract': 'docs/DATASET_SPEC.md v1',
                       'epochs': EPOCHS[tag]},
    }

import shutil

EPOCHS = {'n': 70}
if BEST_S:
    EPOCHS['s'] = 50
source_run = datetime.date.today().isoformat()

# One metadata.json per size describes the deployed (int8) artifact;
# model.onnx stays alongside for fp32 inference.
for tag, onx in onnx_paths.items():
    d = os.path.join(registry_root, f'india-yolov8{tag}')
    os.makedirs(d, exist_ok=True)
    shutil.copy(onx, os.path.join(d, 'model.onnx'))
    shutil.copy(int8_paths[tag], os.path.join(d, 'model-int8.onnx'))
    map50 = mAP50_n if tag == 'n' else mAP50_s
    per_cls = per_class_n if tag == 'n' else per_class_s
    meta = build_meta(f'india-yolov8{tag}', 'int8', source_run, map50, per_cls)
    with open(os.path.join(d, 'metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)

for root_d, _, fs in os.walk(registry_root):
    for fn in fs:
        print(os.path.join(root_d, fn))

models/registry/india-yolov8n/metadata.json
models/registry/india-yolov8n/model.onnx
models/registry/india-yolov8n/model-int8.onnx


## Zip registry and copy back to Drive

Zips `models/registry/`, copies the archive to the Drive dataset root, and
prints download instructions.

In [14]:
import shutil

zip_base = 'india_model_registry'
shutil.make_archive(zip_base, 'zip', root_dir='.', base_dir='models/registry')
zip_path = zip_base + '.zip'
print('created:', zip_path)

from google.colab import files
files.download(zip_path)  # <- the ONLY artifact that leaves Colab

print()
print('Local handoff: unzip so models/registry/india-yolov8*/ lands at repo root,')
print('then run make verify + make eval and update MASTER_PLAN Phase T.')

created: india_model_registry.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Local handoff: unzip so models/registry/india-yolov8*/ lands at repo root,
then run make verify + make eval and update MASTER_PLAN Phase T.


## Handoff checklist

1. The browser downloads `india_model_registry.zip` (previous cell).
2. Unzip its contents into the repo so the layout becomes:

   ```
   models/registry/
     india-yolov8n/{model.onnx, model-int8.onnx, metadata.json}
     india-yolov8s/...   (if trained)
   ```
3. `metadata.json` carries HF provenance (`hf_repo`, `hf_subset`,
   `mapping: optionA-conservative-6class`) — the loader reads classes
   from metadata, never hardcoded.
4. Run the repo test suite (`make verify`, or the pytest equivalent on Windows).
5. Rerun `make eval` to confirm detection metrics against the new artifacts.
6. Update `docs/MASTER_PLAN.md` Phase T status board entry.

If the promotion gate failed in this run: do NOT package — escalate to
Option B (aggressive class merge + BMD-45-Train slice) as a separate run.

Remember: class schema is provisional. New vehicle types = append to end of
`data.yaml`, extend `core/domain.py::VehicleType`, retrain, ship new metadata.
